# OAPR — HRNet Baseline Training on Google Colab (Free T4 GPU)

This notebook trains the **HRNet-W32 baseline** on COCO, then generates **real pose results** (AP metrics + skeleton visualizations) from *your* trained model.

## What you have to do manually (only 2 things)
1. **Enable GPU:** `Runtime → Change runtime type → T4 GPU` → Save.
2. **Authorize Google Drive** when the mount cell prompts you (used to persist checkpoints across sessions).

Then edit the **CONFIG** cell and run cells top-to-bottom (or `Runtime → Run all`).

> Note: This is the **HRNet baseline** path, which is fully wired for COCO images. The `full OAPR` (Mamba + occlusion) model is NOT trainable on COCO as-is and is intentionally not used here.

## Step 1 — Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Set Runtime -> Change runtime type -> T4 GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## Step 2 — CONFIG (edit these)

In [ ]:
# ============ EDIT THESE ============
# Where your project code comes from. EITHER set a GitHub URL...
REPO_URL = 'https://github.com/<your-username>/oapr_pose.git'  # or set '' to use a Drive zip instead
# ...OR upload your project as a zip to Google Drive and set its path here (used only if REPO_URL == ''):
DRIVE_PROJECT_ZIP = '/content/drive/MyDrive/oapr_pose.zip'

# Persistent checkpoint dir on Google Drive (survives Colab disconnects).
DRIVE_CKPT_DIR = '/content/drive/MyDrive/oapr_checkpoints'

# Training data mode:
#   True  = download full train2017 (~19 GB) and train properly (article-grade).
#   False = QUICK preview: train on val2017 only (~1 GB), much faster but weaker.
USE_FULL_TRAIN = True

EPOCHS = 40           # full schedule is 210; 40 is a practical Colab run. Resume to go further.
BATCH_SIZE = 24       # T4 ~15GB. Lower to 16 if you hit CUDA OOM.
EVAL_INTERVAL = 5     # run COCO AP + save best.pth every N epochs
SAVE_FREQ = 5         # save a resumable checkpoint every N epochs
MAX_VIS = 100         # number of qualitative skeleton images to save
# ====================================

## Step 3 — Get your code

Uses GitHub if `REPO_URL` is set; otherwise unzips the project from Drive (mounts Drive first if needed).

In [ ]:
import os
PROJECT_DIR = '/content/oapr_pose'

if REPO_URL:
    if not os.path.exists(PROJECT_DIR):
        get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
else:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    get_ipython().system(f'unzip -q -o "{DRIVE_PROJECT_ZIP}" -d /content/')
    # If the zip extracts to a different folder name, adjust PROJECT_DIR accordingly.

os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
get_ipython().system('ls')

## Step 4 — Install dependencies

We skip `mmcv`, `mamba-ssm`, `causal-conv1d` (not needed for the HRNet baseline and slow/fragile to build on Colab).

In [ ]:
!pip install -q timm pycocotools einops pyyaml tqdm scipy tensorboard opencv-python-headless

## Step 5 — Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', DRIVE_CKPT_DIR)

## Step 6 — Download COCO

Annotations + val2017 always. train2017 (19 GB) only if `USE_FULL_TRAIN = True`.
This can take ~10-20 min the first time (the `-c` flag resumes partial downloads).

In [ ]:
import os
os.makedirs('data/coco/images', exist_ok=True)
sh = get_ipython().system

# Annotations (~250 MB)
if not os.path.exists('data/coco/annotations/person_keypoints_val2017.json'):
    sh('wget -q -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip')
    sh('unzip -q -o annotations_trainval2017.zip -d data/coco')

# Val images (~1 GB) — needed for evaluation/visualization
if not os.path.isdir('data/coco/images/val2017'):
    sh('wget -q -c http://images.cocodataset.org/zips/val2017.zip')
    sh('unzip -q -o val2017.zip -d data/coco/images')

# Train images (~19 GB) — only for full training
if USE_FULL_TRAIN and not os.path.isdir('data/coco/images/train2017'):
    sh('wget -q -c http://images.cocodataset.org/zips/train2017.zip')
    sh('unzip -q -o train2017.zip -d data/coco/images')

print('val images:', len(os.listdir('data/coco/images/val2017')))
if os.path.isdir('data/coco/images/train2017'):
    print('train images:', len(os.listdir('data/coco/images/train2017')))

## Step 7 — Train HRNet baseline

Checkpoints are written to `DRIVE_CKPT_DIR`. If a previous checkpoint exists there, training **auto-resumes** from the latest one. If Colab disconnects, just re-run this cell.

In [ ]:
import glob, os

ckpts = sorted(glob.glob(os.path.join(DRIVE_CKPT_DIR, 'checkpoint_epoch*.pth')))
resume_arg = f'--resume {ckpts[-1]}' if ckpts else ''
if resume_arg:
    print('Resuming from:', ckpts[-1])

override = (
    f'training.epochs={EPOCHS} '
    f'training.batch_size={BATCH_SIZE} '
    f'training.num_workers=2 '
    f'evaluation.interval={EVAL_INTERVAL} '
    f'logging.save_checkpoint_freq={SAVE_FREQ} '
    f'experiment.output_dir={DRIVE_CKPT_DIR}'
)
if not USE_FULL_TRAIN:
    override += ' dataset.train_ann=annotations/person_keypoints_val2017.json'

cmd = f'python train_baseline.py --config configs/baseline_hrnet.yaml {resume_arg} --override {override}'
print(cmd)
get_ipython().system(cmd)

## Step 8 — Generate YOUR model's results (AP + qualitative skeletons)

Runs the trained checkpoint on COCO val: prints **AP / AP50 / AP75** and saves skeleton overlays to `outputs/hrnet_qualitative/`.

In [ ]:
import glob, os

ckpt = os.path.join(DRIVE_CKPT_DIR, 'best.pth')
if not os.path.exists(ckpt):
    cks = sorted(glob.glob(os.path.join(DRIVE_CKPT_DIR, 'checkpoint_epoch*.pth')))
    assert cks, 'No checkpoint found. Train first (Step 7).'
    ckpt = cks[-1]
print('Using checkpoint:', ckpt)

get_ipython().system(
    f'python evaluate.py --config configs/baseline_hrnet.yaml '
    f'--checkpoint {ckpt} --visualize --vis_dir outputs/hrnet_qualitative --max_vis {MAX_VIS}'
)

## Step 9 — Preview a few results inline

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imgs = sorted(glob.glob('outputs/hrnet_qualitative/*.jpg'))[:6]
if imgs:
    plt.figure(figsize=(15, 8))
    for i, p in enumerate(imgs):
        plt.subplot(2, 3, i + 1)
        plt.imshow(mpimg.imread(p))
        plt.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No visualizations found yet.')

## Step 10 — Download results

In [ ]:
import shutil, os
# Also copy a permanent backup to Drive
shutil.make_archive('/content/hrnet_results', 'zip', 'outputs/hrnet_qualitative')
shutil.copy('/content/hrnet_results.zip', os.path.join(DRIVE_CKPT_DIR, 'hrnet_results.zip'))
print('Backed up to Drive:', os.path.join(DRIVE_CKPT_DIR, 'hrnet_results.zip'))

from google.colab import files
files.download('/content/hrnet_results.zip')